In [9]:
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    BartForConditionalGeneration,  # BART's generation model, different from BERT's classification
    AutoModelForSequenceClassification,  # BERT's classification model
    BartTokenizer,
    Seq2SeqTrainer,                # specialized trainer for sequence to sequence tasks like summarization
    Seq2SeqTrainingArguments,      # training arguments for seq2seq models
    DataCollatorForSeq2Seq         # handles padding for seq2seq batches
)
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
# from google.colab import drive

# Mount drive so we can access our CSVs and save our model
# drive.mount('/content/drive')

# Same GPU check as before
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Load our preprocessed splits from Drive
df_train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ToS_summarizer/df_train.csv')
df_val = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ToS_summarizer/df_val.csv')
df_test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ToS_summarizer/df_test.csv')

# We'll use this to convert numeric labels back to human readable strings
# These will be prepended to the input so BART knows what kind of clause it's summarizing
id2label = {
    0: 'clearly_fair',
    1: 'potentially_unfair',
    2: 'clearly_unfair'
}

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print(df_train.head())

# Load the supplemental summarization dataset
summary_dataset = load_dataset("EE21/ToS-Summaries")

# Convert to DataFrame and inspect
df_summaries = pd.DataFrame(summary_dataset['train'])

# See what we're working with
print(df_summaries.shape)
print(df_summaries.columns.tolist())
print(df_summaries.head(3))

Train: 7470 | Val: 971 | Test: 960
                                            sentence    unfairness_level  \
0  these terms and any rights and licenses grante...        clearly_fair   
1  the user is responsible for all damages liabil...      clearly_unfair   
2  no refunds for downtime  the company is not li...  potentially_unfair   
3  ea recommends that parents and guardians famil...        clearly_fair   
4  the company can limit or restrict your ability...  potentially_unfair   

   label  
0      0  
1      2  
2      1  
3      0  
4      1  


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/128 [00:00<?, ?B/s]

dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/901 [00:00<?, ? examples/s]

(901, 2)
['plain_text', 'summary']
                                          plain_text  \
0  We can change these Terms at any time. We keep...   
1  How To File a DMCA Notice To submit a notice o...   
2  You can see our previous Privacy Policy    her...   

                                             summary  
0  Users should revisit the terms periodically, a...  
1  This service will aid you when other users inf...  
2  There is a date of the last update of the agre...  


In [13]:
import os
base_path = os.path.expanduser('~/GitHub-Repos/ToS_summarizer')

In [14]:
# Load our saved BERT model from Drive to label the supplemental dataset
bert_tokenizer = AutoTokenizer.from_pretrained(f'{base_path}/bert_model')
bert_model = AutoModelForSequenceClassification.from_pretrained(f'{base_path}/bert_model')
bert_model = bert_model.to(device)
bert_model.eval()

print("BERT model loaded!")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6542.77it/s]

BERT model loaded!


In [8]:
def run_inference(df, model, tokenizer, device):
    predictions = []

    for sentence in df['sentence']:
        # Tokenize the sentence
        inputs = tokenizer(
            str(sentence),
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to(device)

        # Get prediction
        with torch.no_grad():
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            predictions.append(id2label[pred])

    df['bert_label'] = predictions
    return df

# Rename plain_text to sentence so it matches our existing DataFrames
df_summaries = df_summaries.rename(columns={'plain_text': 'sentence'})

# Run BERT inference on the summaries dataset to get unfairness labels
print("Running BERT inference on summaries dataset...")
df_summaries = run_inference(df_summaries, bert_model, bert_tokenizer, device)

# Sanity check the label distribution
print(df_summaries['bert_label'].value_counts())
print(df_summaries.head(3))

Running BERT inference on summaries dataset...
bert_label
clearly_fair          589
potentially_unfair    258
clearly_unfair         54
Name: count, dtype: int64
                                            sentence  \
0  We can change these Terms at any time. We keep...   
1  How To File a DMCA Notice To submit a notice o...   
2  You can see our previous Privacy Policy    her...   

                                             summary          bert_label  
0  Users should revisit the terms periodically, a...  potentially_unfair  
1  This service will aid you when other users inf...        clearly_fair  
2  There is a date of the last update of the agre...        clearly_fair  


In [9]:
# Load the BART tokenizer
# facebook/bart-large-cnn is pretrained on news summarization
# making it a great starting point for our legal clause summarization
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')

# Load the supplemental summarization dataset
summary_dataset = load_dataset("EE21/ToS-Summaries")
df_summaries = pd.DataFrame(summary_dataset['train'])

# Rename to match our existing DataFrames
df_summaries = df_summaries.rename(columns={'plain_text': 'sentence'})

# Run BERT inference on supplemental dataset to get unfairness labels
print("Running BERT inference on summaries dataset...")
df_summaries = run_inference(df_summaries, bert_model, bert_tokenizer, device)

# Sanity check the label distribution
print(df_summaries['bert_label'].value_counts())
print(df_summaries.head(3))

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Running BERT inference on summaries dataset...
bert_label
clearly_fair          589
potentially_unfair    258
clearly_unfair         54
Name: count, dtype: int64
                                            sentence  \
0  We can change these Terms at any time. We keep...   
1  How To File a DMCA Notice To submit a notice o...   
2  You can see our previous Privacy Policy    her...   

                                             summary          bert_label  
0  Users should revisit the terms periodically, a...  potentially_unfair  
1  This service will aid you when other users inf...        clearly_fair  
2  There is a date of the last update of the agre...        clearly_fair  


In [12]:
# Since sentence text won't match exactly between datasets,
# we'll add the summaries dataset as additional training rows
# and use placeholder summaries for our main dataset

# Add placeholder summaries to main training data
df_train['summary'] = df_train['unfairness_level'] + ': ' + df_train['sentence']
df_val['summary'] = df_val['unfairness_level'] + ': ' + df_val['sentence']

# Prepare supplemental dataset to match our DataFrame structure
df_summaries['unfairness_level'] = df_summaries['bert_label']
df_summaries['label'] = df_summaries['bert_label'].map({
    'clearly_fair': 0,
    'potentially_unfair': 1,
    'clearly_unfair': 2
})

# Append supplemental data to training set — real summaries are valuable training signal
df_train = pd.concat([df_train, df_summaries[['sentence', 'unfairness_level', 'label', 'summary']]], ignore_index=True)

print(f"Train rows after adding summaries: {len(df_train)}")
print(df_train[['sentence', 'summary']].head(3))

Train rows after adding summaries: 8371
                                            sentence  \
0  these terms and any rights and licenses grante...   
1  the user is responsible for all damages liabil...   
2  no refunds for downtime  the company is not li...   

                                             summary  
0  clearly_fair: these terms and any rights and l...  
1  clearly_unfair: the user is responsible for al...  
2  potentially_unfair: no refunds for downtime  t...  


In [14]:
class TOSSummarizationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input_length=256, max_target_length=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        # Get the clause text prefixed with its label as input
        input_text = str(self.data.iloc[index]['sentence'])

        # Get the target summary
        target_text = str(self.data.iloc[index]['summary'])

        # Tokenize the input clause
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Tokenize the target summary
        target_encoding = self.tokenizer(
            target_text,
            max_length=self.max_target_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Replace padding token ids in labels with -100
        # this tells BART to ignore padding tokens when calculating loss
        labels = target_encoding['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels
        }

In [15]:
# Instantiate our dataset class for each split
train_dataset = TOSSummarizationDataset(df_train, tokenizer)
val_dataset = TOSSummarizationDataset(df_val, tokenizer)
test_dataset = TOSSummarizationDataset(df_test, tokenizer)

# DataLoaders handle batching during training
# batch size of 4 because BART is much larger than BERT and needs more memory per sample
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 2093
Val batches: 243
Test batches: 240


In [18]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
# Load the BART model pretrained on CNN/DailyMail news summarization
# This gives us a great starting point since it already knows how to summarize
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

# Move model to GPU if available
model = model.to(device)

# Set up optimizer — same AdamW as BERT but lower learning rate
# BART is bigger and more sensitive so we want smaller updates
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

# Calculate total training steps for the scheduler
total_steps = len(train_loader) * 3  # 3 epochs

# Same warmup scheduler as BERT
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print(f"BART model loaded on: {device}")
print(f"Total training steps: {total_steps}")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

BART model loaded on: cuda
Total training steps: 6279


In [19]:
def train_bart_epoch(model, dataloader, optimizer, scheduler, device):
    # Set model to training mode
    model.train()
    total_loss = 0

    for batch in dataloader:
        # Move batch to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass — BART computes loss internally when labels are provided
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()

        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update weights and scheduler
        optimizer.step()
        scheduler.step()

    return total_loss / len(dataloader)


def eval_bart_epoch(model, dataloader, device):
    # Set model to evaluation mode
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()

    return total_loss / len(dataloader)

In [20]:
EPOCHS = 3
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*50}")

    # Run training pass
    train_loss = train_bart_epoch(
        model, train_loader, optimizer, scheduler, device
    )

    # Run validation pass
    val_loss = eval_bart_epoch(
        model, val_loader, device
    )

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

    # Save model checkpoint if validation loss improved
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained('/content/drive/MyDrive/Colab Notebooks/ToS_summarizer/bart_model')
        tokenizer.save_pretrained('/content/drive/MyDrive/Colab Notebooks/ToS_summarizer/bart_model')
        print(f"✅ Model improved and saved to Drive!")
    else:
        print(f"⚠️ No improvement this epoch")

print("\nTraining complete!")


Epoch 1/3
Train Loss: 0.3377
Val Loss:   0.1821


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model improved and saved to Drive!

Epoch 2/3
Train Loss: 0.1180
Val Loss:   0.1790


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model improved and saved to Drive!

Epoch 3/3
Train Loss: 0.0839
Val Loss:   0.1554


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model improved and saved to Drive!

Training complete!


In [11]:
# Load the saved BART model and tokenizer from Drive
# This ensures we're using the best checkpoint rather than the current training state

bart_model = BartForConditionalGeneration.from_pretrained(f'{base_path}/bart_model')
bart_tokenizer = BartTokenizer.from_pretrained(f'{base_path}/bart_model')

bart_model = bart_model.to(device)
bart_model.eval()

print("✅ BART model loaded from Drive!")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 512/512 [00:00<00:00, 7128.93it/s]


✅ BART model loaded from Drive!


In [22]:
def clean_df(df):
    df = df[df['sentence'].str.len() >= 20].copy()
    df['sentence'] = df['sentence'].str.strip()
    df = df[df['sentence'].str.len() > 0]
    return df

In [23]:
def load_tos_document(file_path):
    # Handles .txt files for now
    # Can be extended to handle .pdf or .docx later
    if file_path.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
    else:
        raise ValueError("Currently only .txt files are supported!")

    # Convert to DataFrame so we can run clean_df on it
    # We split on newlines to get individual lines first
    df = pd.DataFrame({'sentence': [s.strip() for s in text.split('\n') if len(s.strip()) > 0]})

    # Run our preprocessing cleaning function
    df = clean_df(df)

    # Return as a single clean string to pass into process_tos_document
    return ' '.join(df['sentence'].tolist())

# To use with a real document instead of the sample string,
# upload your ToS file to Colab and replace the sample_tos variable like this:
# document = load_tos_document('/content/your_tos_file.txt')
# results = process_tos_document(document, bert_model, model, bert_tokenizer, tokenizer, device)

In [17]:
id2label = {
    0: 'clearly_fair',
    1: 'potentially_unfair',
    2: 'clearly_unfair'
}

In [18]:
import nltk
from nltk.tokenize import sent_tokenize

def process_tos_document(document, bert_model, bart_model, bert_tokenizer, bart_tokenizer, device):
    # Step 1 — Split document into individual clauses by sentence
    # We use a simple split on punctuation for now
    raw_sentences = sent_tokenize(document.replace('\n', ' '))
    clauses = [s.strip() for s in raw_sentences if len(s.strip()) >= 20]

    print(f"Found {len(clauses)} clauses in document")

    # Step 2 — Run BERT classification on each clause
    bert_model.eval()
    flagged_clauses = []

    for clause in clauses:
        inputs = bert_tokenizer(
            clause,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = bert_model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            label = id2label[pred]

        # Step 3 — Filter out clearly fair clauses, keep the rest
        if label != 'clearly_fair':
            flagged_clauses.append((clause, label))

    print(f"Flagged {len(flagged_clauses)} potentially risky clauses")

    # Step 4 — Run BART summarization on flagged clauses
    bart_model.eval()
    results = []

    for clause, label in flagged_clauses:
        inputs = bart_tokenizer(
            clause,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            # Generate summary
            summary_ids = bart_model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_length=128,
                min_length=20,
                num_beams=4,        # beam search for better quality output
                length_penalty=2.0, # encourages longer summaries
                early_stopping=True
            )

        summary = bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        # Strip any label prefixes that leaked from placeholder training data
        for prefix in ['clearly_fair: ', 'clearly_unfair: ', 'potentially_unfair: ']:
            summary = summary.replace(prefix, '')

        results.append((label, summary))

    # Step 5 — Format and print plain English output
    print("\n" + "="*50)
    print("TERMS OF SERVICE ANALYSIS")
    print("="*50)

    for label, summary in results:
        if label == 'clearly_unfair':
            emoji = '🚨'
        else:
            emoji = '⚠️'

        print(f"\n{emoji} {label.upper().replace('_', ' ')}")
        print(f"• {summary}")

    print("\n" + "="*50)
    return results


# Test it with a sample clause once BART finishes training
sample_tos = """
The company reserves the right to terminate your account at any time without notice.
Users are responsible for all charges incurred under their account.
We may share your personal data with third party partners for marketing purposes.
You agree to receive promotional emails from us and our partners.
"""

results = process_tos_document(
    sample_tos,
    bert_model,
    bart_model,  # this is our BART model
    bert_tokenizer,
    bart_tokenizer,  # this is our BART tokenizer
    device
)

Found 4 clauses in document
Flagged 2 potentially risky clauses

TERMS OF SERVICE ANALYSIS

⚠️ POTENTIALLY UNFAIR
• the company reserves the right to terminate your account at any time without notice.

🚨 CLEARLY UNFAIR
• users are responsible for all charges incurred under their account. The service is not responsible for linked or (clearly) quoted content from third party content providers.



In [21]:
# Load and process a real ToS document
# document = load_tos_document(f'{base_path}/fb_tos_testing.txt')
with open(f'{base_path}/fb_tos_testing.txt', 'r') as f:
    document = f.read()
# Run the full pipeline!
results = process_tos_document(
    document,
    bert_model,
    bart_model,
    bert_tokenizer,
    bart_tokenizer,
    device
)

Found 184 clauses in document
Flagged 38 potentially risky clauses

TERMS OF SERVICE ANALYSIS

🚨 CLEARLY UNFAIR
• these Terms therefore constitute an agreement between you and Meta Platforms, Inc.

🚨 CLEARLY UNFAIR
•  Terms may be changed any time at their discretion, without notice to the user . This service assumes no liability for any losses or damages resulting from any matter relating to the service. The service is provided 'as is' and to be used at the users' sole risk. This service does not guarantee that it or the products obtained through it meet your expectations or requirements. The court of law governing the terms is in a jurisdiction that is friendlier to user privacy protection (Switzerland).

🚨 CLEARLY UNFAIR
• The service provides details about what kinds of personal information they collect. The service provides information about how they intend to use your personal data. This service employs third-party cookies, but with opt-out instructions. 

🚨 CLEARLY UNFAIR
• What